# Chapter 2 — Delivery Performance
Queries `mart_delivery_performance` from BigQuery and exports Plotly chart JSON for the webpage.

In [1]:
from dotenv import load_dotenv
import os, json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery
from google.oauth2 import service_account

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(project_root, '.env'))

project_id  = os.getenv('GCP_PROJECT_ID')
creds_path  = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials = service_account.Credentials.from_service_account_file(creds_path)
client      = bigquery.Client(credentials=credentials, project=project_id)

OUT = os.path.join(project_root, 'outputs')
os.makedirs(OUT, exist_ok=True)
print('Connected to BigQuery ✓')

Connected to BigQuery ✓


In [2]:
df = client.query(f"""
    SELECT * FROM `{project_id}.olist_raw.mart_delivery_performance`
    ORDER BY year_month, seller_state
""").to_dataframe()
df.head()

,seller_state,year_month,year,month,total_orders,on_time_deliveries,late_deliveries,not_delivered,on_time_pct,avg_delivery_days,avg_review_score,avg_days_late
0,PR,2016-09,2016,9,1,0,3,0,0.0,55.0,1.00,36.0
1,BA,2016-10,2016,10,1,1,0,0,100.0,21.0,5.00,NaN
2,DF,2016-10,2016,10,2,2,0,0,100.0,10.0,2.50,NaN
3,ES,2016-10,2016,10,1,1,0,0,100.0,25.0,5.00,NaN
4,MG,2016-10,2016,10,21,25,0,0,100.0,17.8,3.52,NaN


In [3]:
# Chart 1 — On-Time % by Seller State (aggregated across all months)
state_summary = df.groupby('seller_state').agg(
    total_orders=('total_orders', 'sum'),
    on_time=('on_time_deliveries', 'sum'),
    late=('late_deliveries', 'sum'),
    avg_days=('avg_delivery_days', 'mean'),
    avg_score=('avg_review_score', 'mean')
).reset_index()
state_summary['on_time_pct'] = (state_summary['on_time'] / (state_summary['on_time'] + state_summary['late']) * 100).round(1)
state_summary = state_summary.sort_values('on_time_pct', ascending=True)

fig1 = px.bar(
    state_summary, x='on_time_pct', y='seller_state', orientation='h',
    title='On-Time Delivery % by Seller State',
    labels={'on_time_pct': 'On-Time %', 'seller_state': 'State'},
    color='on_time_pct',
    color_continuous_scale='RdYlGn'
)
fig1.update_layout(template='plotly_white', coloraxis_showscale=False)
fig1.show()
with open(os.path.join(OUT, 'delivery_on_time_by_state.json'), 'w') as f:
    f.write(fig1.to_json())
print('Exported delivery_on_time_by_state.json')

Exported delivery_on_time_by_state.json


In [4]:
# Chart 2 — Average Delivery Days by State
state_days = state_summary.sort_values('avg_days', ascending=False)
fig2 = px.bar(
    state_days, x='seller_state', y='avg_days',
    title='Average Delivery Days by Seller State',
    labels={'seller_state': 'State', 'avg_days': 'Avg Delivery Days'},
    color='avg_days',
    color_continuous_scale='Blues'
)
fig2.update_layout(template='plotly_white', coloraxis_showscale=False)
fig2.show()
with open(os.path.join(OUT, 'delivery_avg_days_by_state.json'), 'w') as f:
    f.write(fig2.to_json())
print('Exported delivery_avg_days_by_state.json')

Exported delivery_avg_days_by_state.json


In [5]:
# Chart 3 — Monthly On-Time % trend
monthly = df.groupby('year_month').agg(
    on_time=('on_time_deliveries', 'sum'),
    late=('late_deliveries', 'sum')
).reset_index()
monthly['on_time_pct'] = (monthly['on_time'] / (monthly['on_time'] + monthly['late']) * 100).round(1)

fig3 = px.line(
    monthly, x='year_month', y='on_time_pct',
    title='Monthly On-Time Delivery Rate (%)',
    labels={'year_month': 'Month', 'on_time_pct': 'On-Time %'},
    markers=True,
    color_discrete_sequence=['#2ecc71']
)
fig3.update_layout(template='plotly_white', hovermode='x unified')
fig3.show()
with open(os.path.join(OUT, 'delivery_monthly_trend.json'), 'w') as f:
    f.write(fig3.to_json())
print('Exported delivery_monthly_trend.json')

Exported delivery_monthly_trend.json


In [6]:
# Chart 4 — Review Score vs On-Time % by State (bubble)
fig4 = px.scatter(
    state_summary, x='on_time_pct', y='avg_score',
    size='total_orders', text='seller_state',
    title='Review Score vs On-Time Delivery % by State',
    labels={'on_time_pct': 'On-Time %', 'avg_score': 'Avg Review Score', 'total_orders': 'Total Orders'},
    color='avg_score',
    color_continuous_scale='RdYlGn'
)
fig4.update_traces(textposition='top center')
fig4.update_layout(template='plotly_white')
fig4.show()
with open(os.path.join(OUT, 'delivery_score_vs_ontime.json'), 'w') as f:
    f.write(fig4.to_json())
print('Exported delivery_score_vs_ontime.json')

Exported delivery_score_vs_ontime.json
